# Danh gia mo hinh – Multilingual MT
Sinh ra cac metric: **Val Loss, Perplexity, BLEU Score** cho ca 2 mo hinh.

**Yeu cau truoc khi chay:**
- Dat `best_transformer_model.pt` vao thu muc `model_assets/`
- Dat `best_lstm_model.pt` vao thu muc `model_assets/` (neu co)
- Chay kernel: `Python (Multilingual MT)`

## Cell 1 – Setup duong dan & thu vien

In [2]:
import os, sys, math

# Thu muc goc du an (tu dong tim)
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Cai sacrebleu neu chua co
try:
    import sacrebleu
except ImportError:
    import subprocess; subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sacrebleu', '-q'])
    import sacrebleu

import torch
from tqdm import tqdm
from tokenizers import Tokenizer as HFTok

# Duong dan
MODEL_DIR      = os.path.join(PROJECT_DIR, 'model_assets')
TF_CKPT        = os.path.join(MODEL_DIR,   'best_transformer_model.pt')
LSTM_CKPT      = os.path.join(MODEL_DIR, 'best_baseline_model.pt')
TEST_FILE      = os.path.join(PROJECT_DIR, 'data', 'processed', 'test.txt')
TOKENIZER_PATH = os.path.join(PROJECT_DIR, 'tokenizer', 'tokenizer.json')

# Token IDs
PAD_IDX = 0
BOS_IDX = 2
EOS_IDX = 3
MAX_LEN = 64

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Project dir : {PROJECT_DIR}')
print(f'Device      : {DEVICE}')
print()
for name, path in [('Transformer .pt', TF_CKPT), ('LSTM .pt', LSTM_CKPT),
                   ('test.txt', TEST_FILE), ('tokenizer.json', TOKENIZER_PATH)]:
    status = 'OK' if os.path.exists(path) else 'THIEU'
    size   = f'({os.path.getsize(path)/1e6:.1f}MB)' if os.path.exists(path) else ''
    print(f'  {status:5} : {name} {size}')

Project dir : c:\PKA\Neuron network\Multilingual_MT
Device      : cpu

  OK    : Transformer .pt (263.1MB)
  OK    : LSTM .pt (481.9MB)
  OK    : test.txt (5.2MB)
  OK    : tokenizer.json (1.9MB)


## Cell 2 – Load model Transformer

In [3]:
from src.models.transformer import Transformer

tf_ckpt = torch.load(TF_CKPT, map_location=DEVICE)
tf_cfg  = tf_ckpt['model_config']

transformer = Transformer(**tf_cfg).to(DEVICE)
transformer.load_state_dict(tf_ckpt['model_state'])
transformer.eval()

tf_val_loss = tf_ckpt['val_loss']
tf_ppl      = math.exp(tf_val_loss)

print(f'Transformer da load:')
print(f'  Epoch     : {tf_ckpt["epoch"]}')
print(f'  Val Loss  : {tf_val_loss:.4f}')
print(f'  Perplexity: {tf_ppl:.2f}')
total = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
print(f'  Params    : {total:,}')

Transformer da load:
  Epoch     : 10
  Val Loss  : 4.8150
  Perplexity: 123.34
  Params    : 21,905,408


## Cell 3 – Load model LSTM Baseline (bo qua neu khong co file)

In [4]:
from src.models.baseline_lstm import LSTMBaseline

lstm_model    = None
lstm_val_loss = 4.25     # Gia tri da biet tu log train
lstm_ppl      = math.exp(lstm_val_loss)
lstm_bleu     = None     # Se tinh o cell sau neu co file

if os.path.exists(LSTM_CKPT):
    lstm_ckpt = torch.load(LSTM_CKPT, map_location=DEVICE)
    lstm_cfg  = lstm_ckpt['model_config']
    lstm_model = LSTMBaseline(**lstm_cfg).to(DEVICE)
    lstm_model.load_state_dict(lstm_ckpt['model_state'])
    lstm_model.eval()
    lstm_val_loss = lstm_ckpt['val_loss']
    lstm_ppl      = math.exp(lstm_val_loss)
    print(f'LSTM Baseline da load:')
    print(f'  Epoch     : {lstm_ckpt["epoch"]}')
    print(f'  Val Loss  : {lstm_val_loss:.4f}')
    print(f'  Perplexity: {lstm_ppl:.2f}')
else:
    print('Khong tim thay best_lstm_model.pt')
    print(f'Su dung gia tri da biet: Val Loss={lstm_val_loss}, PPL={lstm_ppl:.2f}')

LSTM Baseline da load:
  Epoch     : 10
  Val Loss  : 4.2456
  Perplexity: 69.80


## Cell 4 – Ham dich (Greedy Decode)

In [5]:
tok = HFTok.from_file(TOKENIZER_PATH)

def translate_transformer(src_text: str) -> str:
    ids  = tok.encode(src_text).ids[:MAX_LEN]
    ids += [PAD_IDX] * (MAX_LEN - len(ids))
    src_t = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        out = transformer.translate_greedy(src_t, BOS_IDX, EOS_IDX, MAX_LEN)
    return tok.decode(out, skip_special_tokens=True)


def translate_lstm(src_text: str) -> str:
    if lstm_model is None:
        return '[LSTM model khong co san]'
    ids  = tok.encode(src_text).ids[:MAX_LEN]
    ids += [PAD_IDX] * (MAX_LEN - len(ids))
    src_t = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        # Encoder
        enc_h, enc_c = lstm_model.encoder(src_t)
        hidden = (enc_h, enc_c)

        # Greedy decode: truy cap truc tiep cac tang ben trong Decoder
        dec_input = torch.tensor([[BOS_IDX]], dtype=torch.long).to(DEVICE)
        out_ids   = []
        dec = lstm_model.decoder   # tham chieu den Decoder module

        for _ in range(MAX_LEN):
            embedded        = dec.embedding(dec_input)          # [1, 1, E]
            output, hidden  = dec.lstm(embedded, hidden)        # [1, 1, H], cap nhat hidden
            logit           = dec.output_projection(output)     # [1, 1, V]
            next_id         = logit[0, -1].argmax().item()
            if next_id == EOS_IDX:
                break
            out_ids.append(next_id)
            dec_input = torch.tensor([[next_id]], dtype=torch.long).to(DEVICE)

    return tok.decode(out_ids, skip_special_tokens=True)



# Thu nhanh
samples = [
    '<2vi> Hello , how are you ?',
    '<2ja> I love machine learning .',
    '<2zh> Thank you very much .',
]
print('Thu dich nhanh:\n')
for s in samples:
    print(f'  SRC  : {s}')
    print(f'  TF   : {translate_transformer(s)}')
    print(f'  LSTM : {translate_lstm(s)}')
    print()

Thu dich nhanh:

  SRC  : <2vi> Hello , how are you ?
  TF   : Xin chào , bạn là gì ?
  LSTM : Xin chào , thế nào ?

  SRC  : <2ja> I love machine learning .
  TF   : 私は 科学 者 だった
  LSTM : 私は 学 術 学 の 学 んだ

  SRC  : <2zh> Thank you very much .
  TF   : 谢谢 你
  LSTM : 谢谢 你的



## Cell 5 – Tinh BLEU Score tren test.txt

In [6]:
def compute_bleu(translate_fn, test_file, label='Model'):
    refs, hyps = [], []
    with open(test_file, encoding='utf-8') as f:
        lines = [l.strip() for l in f if '\t' in l.strip()]
    for line in tqdm(lines, desc=f'  [{label}] Dang dich'):
        src, tgt = line.split('\t', 1)
        hyps.append(translate_fn(src))
        refs.append(tgt)
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    return bleu.score, hyps, refs


print('Tinh BLEU cho Transformer...')
tf_bleu, tf_hyps, tf_refs = compute_bleu(translate_transformer, TEST_FILE, 'Transformer')
print(f'  Transformer BLEU: {tf_bleu:.2f}\n')

if lstm_model is not None:
    print('Tinh BLEU cho LSTM Baseline...')
    lstm_bleu, lstm_hyps, _ = compute_bleu(translate_lstm, TEST_FILE, 'LSTM')
    print(f'  LSTM Baseline BLEU: {lstm_bleu:.2f}')
else:
    lstm_bleu = None
    print('Bo qua BLEU cho LSTM (khong co file .pt)')

Tinh BLEU cho Transformer...


  [Transformer] Dang dich:   0%|          | 127/30000 [00:34<2:14:52,  3.69it/s]


KeyboardInterrupt: 

## Cell 6 – Bang so sanh chinh thuc

In [10]:
import pandas as pd

lstm_bleu_str = f'{lstm_bleu:.2f}' if lstm_bleu is not None else 'N/A'
tf_better     = tf_val_loss < lstm_val_loss

rows = [
    ['Val Loss (thap hon = tot hon)', f'{lstm_val_loss:.4f}', f'{tf_val_loss:.4f}',
     'Transformer' if tf_better else 'LSTM'],
    ['Perplexity (thap hon = tot hon)', f'{lstm_ppl:.2f}', f'{tf_ppl:.2f}',
     'Transformer' if tf_ppl < lstm_ppl else 'LSTM'],
    ['BLEU Score (cao hon = tot hon)', lstm_bleu_str, f'{tf_bleu:.2f}',
     'Transformer' if lstm_bleu is None or tf_bleu > lstm_bleu else 'LSTM'],
]

df = pd.DataFrame(rows, columns=['Metric', 'LSTM Baseline', 'Transformer', 'Model tot hon'])
print('\n' + '='*65)
print('  BANG SO SANH HIEU NANG 2 MO HINH')
print('='*65)
print(df.to_string(index=False))
print('='*65)
print(f'\n  Ket luan: Transformer cai thien Val Loss -{lstm_val_loss - tf_val_loss:.4f}')
print(f'            va Perplexity -{lstm_ppl - tf_ppl:.2f} so voi LSTM Baseline.')


  BANG SO SANH HIEU NANG 2 MO HINH
                         Metric LSTM Baseline Transformer Model tot hon
  Val Loss (thap hon = tot hon)        4.2456      4.0785   Transformer
Perplexity (thap hon = tot hon)         69.80       59.05   Transformer
 BLEU Score (cao hon = tot hon)          2.51       14.55   Transformer

  Ket luan: Transformer cai thien Val Loss -0.1672
            va Perplexity -10.75 so voi LSTM Baseline.


## Cell 7 – Phan tich loi (Error Analysis)

In [14]:
import sacrebleu

sentence_scores = [
    (sacrebleu.sentence_bleu(hyp, [ref]).score, hyp, ref)
    for hyp, ref in zip(tf_hyps, tf_refs)
]

worst = sorted(sentence_scores, key=lambda x: x[0])[:5]
best  = sorted(sentence_scores, key=lambda x: x[0], reverse=True)[:5]

print('5 CAU DICH TOT NHAT (Transformer)\n')
for i, (score, hyp, ref) in enumerate(best, 1):
    print(f'  [{i}] BLEU={score:.1f}')
    print(f'       REF : {ref}')
    print(f'       HYP : {hyp}')
    print()

print('\n5 CAU DICH TE NHAT (Transformer)\n')
for i, (score, hyp, ref) in enumerate(worst, 1):
    print(f'  [{i}] BLEU={score:.1f}')
    print(f'       REF : {ref}')
    print(f'       HYP : {hyp}')
    print()


5 CAU DICH TOT NHAT (Transformer)

  [1] BLEU=100.0
       REF : Cảm ơn .
       HYP : Cảm ơn .

  [2] BLEU=100.0
       REF : たぶん
       HYP : たぶん

  [3] BLEU=100.0
       REF : 大 会
       HYP : 大 会

  [4] BLEU=100.0
       REF : 愛してる
       HYP : 愛してる

  [5] BLEU=100.0
       REF : - 药...
       HYP : - 药 ...


5 CAU DICH TE NHAT (Transformer)

  [1] BLEU=0.0
       REF : 1人の怒れる父親の姿があるのみ
       HYP : 警察 は ゾ ー イ に 訴 え られた

  [2] BLEU=0.0
       REF : そんなんじゃ・・
       HYP : 彼女の 事は 心配 です

  [3] BLEU=0.0
       REF : 现在很多家庭都面临着压力
       HYP : 今天 有 很多 人 áp lực

  [4] BLEU=0.0
       REF : 伊维！
       HYP : E vie !

  [5] BLEU=0.0
       REF : -我不看女人的眼睛.
       HYP : - 我 眼睛 看到 女人

